In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import os


# 1. SETUP & LOAD DATA

# Use the correct path to your file
file_path = "final_feature_dataset_selected.csv"  # Or "../data/processed/final_feature_dataset_selected.csv"

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f" Data Loaded. Shape: {df.shape}")
else:
    print(f" Error: File not found at {file_path}")
    exit()


# 2. DEFINE COMPLEXITY FEATURES


complexity_features = [
    "num_links",
    "num_images",           
    "num_iframes",          
    "num_forms",           
    "num_inputs",           
    "num_buttons",          
    "num_scripts",
    "has_unescape",        
    "num_styles",           
    "css_external_links",   
    "num_hidden_elements"   
]

# Double-check that these columns actually exist
existing_features = [f for f in complexity_features if f in df.columns]
missing_features = set(complexity_features) - set(existing_features)

if missing_features:
    print(f" Warning: The following features were not found and skipped: {missing_features}")

print(f" Using {len(existing_features)} features for complexity scoring.")


# 3. NORMALIZE & CALCULATE SCORE

scaler = MinMaxScaler()

# Create a temporary dataframe for scaling (so we don't mess up the original values)
df_scaled = pd.DataFrame(
    scaler.fit_transform(df[existing_features]),
    columns=existing_features
)

# Compute Complexity Score: Sum of all normalized structural features
df["complexity_score"] = df_scaled.sum(axis=1)


# 4. CATEGORIZE (Simple / Medium / Hard)

# qcut splits the data into 3 equal buckets (Quantile-based binning)
df["category"] = pd.qcut(df["complexity_score"], q=3, labels=["Simple", "Medium", "Hard"])


# 5. SAVE RESULT

output_filename = "webpage_complexity_categorized_weighted.csv"
df.to_csv(output_filename, index=False)

print(f"\n Categorization completed! Saved as '{output_filename}'")
print("\nDistribution of Categories:")
print(df["category"].value_counts())


: 